In [ ]:
import pipeline


In [ ]:
import requests
import json
import csv
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import aiohttp
import asyncio
import pandas as pd
from tqdm.asyncio import tqdm_asyncio

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
import yaml

# Open the YAML file and load its contents
with open("../config/settings.yaml", "r") as file:
    # safe_load prevents execution of arbitrary code embedded in YAML files
    config = yaml.safe_load(file)

In [ ]:
team = config["team"]
season = config["season"]

print(team)
print(season)


In [ ]:
SCHEDULE_URL = "https://api-web.nhle.com/v1/club-schedule-season/{}/{}"

async def fetch_team_scheldule(session, team, season, sem, retries=3):
    url = SCHEDULE_URL.format(team, season)
    async with sem:  # throttle concurrency
        for attempt in range(retries):
            try:
                async with session.get(url, timeout=10) as resp:
                    if resp.status == 200:
                        return await resp.json()
                    elif resp.status == 404:
                        return None  # Team not found
            except Exception:
                if attempt == retries - 1:
                    return None
                await asyncio.sleep(0.5)

async def fetch_all_games(concurrency=10):
    sem = asyncio.Semaphore(concurrency)
    timeout = aiohttp.ClientTimeout(total=30)

    async with aiohttp.ClientSession(timeout=timeout) as session:
        tasks = [
            fetch_team_scheldule(session, team, season, sem)
        ]
        results = await tqdm_asyncio.gather(*tasks)
        return results
    


In [ ]:
team_data = await fetch_all_games()

In [ ]:
DET_games = team_data[0]["games"]

In [ ]:
DET_games[7]

In [ ]:
game_ids = []
for game in DET_games:
    for key, value in game.items():
        if key == "id":
            game_ids.append(value)

In [ ]:
game_ids[0]

In [ ]:
PBP_URL = "https://api-web.nhle.com/v1/gamecenter/{}/play-by-play"

async def fetch_team_pbp(session, game_id, sem, retries=3):
    url = PBP_URL.format(game_id)
    async with sem:  # throttle concurrency
        for attempt in range(retries):
            try:
                async with session.get(url, timeout=10) as resp:
                    if resp.status == 200:
                        return await resp.json()
                    elif resp.status == 404:
                        return None  # pbp not found
            except Exception:
                if attempt == retries - 1:
                    return None
                await asyncio.sleep(0.5)

async def fetch_all_pbp(game_ids, concurrency=3):
    sem = asyncio.Semaphore(concurrency)
    timeout = aiohttp.ClientTimeout(total=30)

    async with aiohttp.ClientSession(timeout=timeout) as session:
        tasks = [
            fetch_team_pbp(session, gid, sem)
            for gid in game_ids 
        ]
        results = await tqdm_asyncio.gather(*tasks)
        return results

In [ ]:
game_ids = list(game_ids)

In [ ]:
game_ids

In [ ]:
pbp_data = await fetch_all_pbp(game_ids)

In [ ]:
pbp_data

In [ ]:
pbp_data[0]